In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath('..'))

# Set styling for plots
sns.set_theme(style="whitegrid")

# End-to-End Evaluation of a Safety-Aware Adaptive Pricing Engine

This notebook evaluates the pricing engine as a complete decision-making system
prior to any performance benchmarking or deployment.

The objective is not to maximize headline metrics, but to verify that the system
is statistically sound, behaviorally safe, and robust to realistic failure modes.

Evaluation is conducted offline and in simulated online settings, using strictly
out-of-sample data and explicit safety constraints. Only if all evaluation criteria
are satisfied does the system proceed to benchmarking.



The pricing engine is composed of four independent subsystems:

• Demand estimation models that output booking probabilities  
• Adaptive pricing policies that select prices under uncertainty  
• A safety governor that enforces hard business and regulatory constraints  
• An evaluation layer that audits correctness, stability, and trustworthiness  

This notebook evaluates the interaction of these subsystems, rather than any
single model in isolation.


The evaluation follows three core principles:

1. Separation of concerns  
   Learning, safety, and evaluation are treated as independent systems.

2. Out-of-sample correctness  
   No model or policy is evaluated on data it was trained on.

3. Failure-first design  
   The evaluation prioritizes identifying unsafe or misleading behavior over
   reporting optimistic performance.


The output of this notebook is a binary decision:

• GO   — the pricing engine is correct and robust enough to be benchmarked  
• NO-GO — the system requires further modeling or safety iteration  

This decision is based on statistical validity, safety correctness, learning
behavior, and robustness under stress.


In [2]:
import numpy as np
import random

np.random.seed(42)
random.seed(42)

print("Evaluation environment initialized.")


Evaluation environment initialized.


We load the pricing engine as a unified module. All evaluation components operate
on this shared implementation to ensure consistency across experiments.


In [3]:
import pricing_engine as pe

print("Pricing engine loaded.")


Pricing engine loaded.


## Data Integrity, Feature Contract & Temporal Split

The evaluation operates on historical booking data prepared by the data loading
pipeline. During loading, prices for booked nights are recovered using
listing-level temporal imputation to ensure that all realized bookings have valid
prices.

This preprocessing step is essential for counterfactual revenue estimation and
is treated as part of data engineering rather than evaluation. The evaluation
therefore validates assumptions on the prepared data, without reapplying or
replicating the cleaning logic.


In [4]:
from pathlib import Path



try:
    SCRIPT_DIR = Path(__file__).parent
except NameError:
    SCRIPT_DIR = Path.cwd()

# SCRIPT_DIR is the 'notebooks' folder.
# SCRIPT_DIR.parent is the 'Dynamic Pricing Engine' folder.
PROJECT_ROOT = SCRIPT_DIR.parent

# Now build the path from the project root
CALENDAR_PATH = PROJECT_ROOT / 'data' / 'calendar.csv'
LISTINGS_PATH = PROJECT_ROOT / 'data' / 'listings.csv'



In [5]:
df = pe.load_and_clean_seattle_data(CALENDAR_PATH, LISTINGS_PATH)
df.head()

,listing_id,date,available,price,is_booked,neighborhood
0,3335,2016-01-04,f,120.0,1,Rainier Valley
1,3335,2016-01-05,f,120.0,1,Rainier Valley
2,3335,2016-01-06,f,120.0,1,Rainier Valley
3,3335,2016-01-07,f,120.0,1,Rainier Valley
4,3335,2016-01-08,f,120.0,1,Rainier Valley


Before feature construction or model training, we verify that the prepared dataset
satisfies the minimal structural and domain assumptions required for offline
evaluation.


In [6]:
required_columns = {"price", "is_booked", "date", "neighborhood", "listing_id"}

assert required_columns.issubset(df.columns), "Required columns missing"
assert df["price"].min() > 0, "Non-positive prices detected"
assert set(df["is_booked"].unique()).issubset({0, 1}), "Invalid booking labels"

print("Prepared data passed structural and domain checks.")


Prepared data passed structural and domain checks.


Model features are derived explicitly from the prepared dataset to ensure
transparency, reproducibility, and consistency across all demand models and pricing
policies.

All feature transformations are applied prior to any model training or evaluation.


In [7]:
# Calendar-based features
df["day_of_week"] = df["date"].dt.weekday          # 0 = Monday
df["month"] = df["date"].dt.month

# Weekend indicator (Saturday = 5, Sunday = 6)
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

print("Temporal features constructed.")


Temporal features constructed.


Demand models require purely numeric inputs. Categorical variables are therefore
encoded explicitly at the evaluation level to ensure a stable and reproducible
feature interface.

Encoding is deterministic and frozen for the remainder of the evaluation.


In [8]:
# Encode neighborhood as numeric codes (deterministic)
df["neighborhood_code"] = (
    df["neighborhood"]
    .astype("category")
    .cat.codes
)

df[["neighborhood", "neighborhood_code"]].head()


,neighborhood,neighborhood_code
0,Rainier Valley,13
1,Rainier Valley,13
2,Rainier Valley,13
3,Rainier Valley,13
4,Rainier Valley,13


All demand models and pricing policies operate on the same explicit feature set.
This feature contract is frozen for the remainder of the evaluation to prevent
information leakage and ensure meaningful comparisons.


In [9]:
feature_cols = [
    "neighborhood_code",
    "day_of_week",
    "is_weekend",
    "month",
]

missing_features = set(feature_cols) - set(df.columns)
assert not missing_features, f"Missing features after engineering: {missing_features}"

print(f"Feature contract established with {len(feature_cols)} features:")
feature_cols


Feature contract established with 4 features:


['neighborhood_code', 'day_of_week', 'is_weekend', 'month']

Pricing systems operate in non-stationary environments. To reflect real deployment
conditions and avoid optimistic bias, we adopt a time-aware evaluation protocol.

Demand models are trained exclusively on historical data and evaluated only on
future observations. No model or policy is ever trained or tuned on evaluation data.


In [10]:
df = df.sort_values("date")

split_point = df["date"].quantile(0.8)

train_df = df[df["date"] <= split_point].copy()
eval_df  = df[df["date"] > split_point].copy()

print("Training period :", train_df["date"].min(), "→", train_df["date"].max())
print("Evaluation period:", eval_df["date"].min(),  "→", eval_df["date"].max())
print("Train rows:", len(train_df), "| Eval rows:", len(eval_df))


Training period : 2016-01-04 00:00:00 → 2016-10-21 00:00:00
Evaluation period: 2016-10-22 00:00:00 → 2017-01-02 00:00:00
Train rows: 1114856 | Eval rows: 278714


The evaluation dataset represents future market conditions relative to the training
data. All subsequent analysis—including calibration, counterfactual value
estimation, learning dynamics, and stress testing—operates exclusively on this
held-out set.


In [11]:
assert len(train_df) > 0 and len(eval_df) > 0, "Empty split detected"
assert train_df["date"].max() < eval_df["date"].min(), "Temporal leakage detected"

print("Temporal split validated and evaluation protocol locked.")


Temporal split validated and evaluation protocol locked.


## Demand Model Training & Probabilistic Trust Gate

All pricing decisions in the engine are driven by probabilistic estimates of booking
likelihood. Before evaluating any pricing policy, we must therefore establish that
these probability estimates are statistically trustworthy.

In this step, a single demand model is trained on historical data and evaluated
out-of-sample on future observations. This mirrors real deployment, where one
probability oracle is consumed by all pricing policies.


The pricing engine relies on a hierarchical Bayesian logistic demand model that
shares statistical strength across listings while allowing local adaptation.

Policy evaluation is therefore conditioned on this demand model alone. Alternative
demand models may be evaluated in separate runs, but probabilities are never mixed
within a single evaluation.


In [12]:
from pricing_engine import HierarchicalBayesianLogisticDemand

BETA_PRICE = -2.0  # example value; fixed a priori

demand_model = HierarchicalBayesianLogisticDemand(
    beta_price=BETA_PRICE,
    min_listing_obs=30
)

print("Hierarchical Bayesian demand model instantiated.")


Hierarchical Bayesian demand model instantiated.


The demand model is trained exclusively on the training dataset derived from
historical data. Once trained, model parameters are frozen and no retraining or
tuning occurs during evaluation.

This separation between training and evaluation is critical for honest assessment
of downstream pricing policies.


In [13]:
demand_model.fit(
    train_df,
    feature_cols
)

print(f"Demand model trained on {len(train_df)} observations.")


Demand model trained on 1114856 observations.


A demand model may achieve good average accuracy while still producing unreliable
probability estimates. Because pricing policies act directly on predicted booking
probabilities, calibration is evaluated explicitly.

We assess probabilistic correctness using the Brier score and Expected Calibration
Error (ECE). Poor calibration invalidates counterfactual policy evaluation and
constitutes a hard stop.


In [14]:
from pricing_engine.evaluation.offline.demand_calibration import DemandCalibrationEvaluator

calibrator = DemandCalibrationEvaluator(n_bins=10)

cal_result = calibrator.evaluate(
    demand_model,
    eval_df
)

print(
    f"{cal_result.model_name} | "
    f"Brier Score: {cal_result.brier_score:.4f} | "
    f"ECE: {cal_result.ece:.4f}"
)


HierarchicalBayesianLogisticDemand | Brier Score: 0.2021 | ECE: 0.0408


Only demand models that satisfy calibration requirements are eligible for policy
evaluation. If this gate fails, downstream counterfactual estimates are not
statistically meaningful.


In [15]:
MAX_ECE = 0.05

calibration_ok = cal_result.ece <= MAX_ECE

print("Calibration gate passed:", calibration_ok)


Calibration gate passed: True


## Safety-Aware Counterfactual Policy Value Estimation

We now evaluate adaptive pricing policies as they would operate in production.

Policies are stateful learning agents that:

• consume probabilistic demand estimates from a fixed demand model

• update online based on observed outcomes

• operate under strict safety constraints

Evaluation is performed via sequential replay of historical data, ensuring that
learning dynamics, safety behavior, and economic value are assessed jointly.


Bandit policies treat the demand model as a black-box prior that maps feature vectors
to booking probability estimates with uncertainty.

This interface ensures that policies remain model-agnostic and do not inspect
internal demand model parameters.


In [16]:
# ------------------------------------------------------------
# Precompute demand probability & uncertainty (O(N))
# ------------------------------------------------------------
eval_df = eval_df.copy()

p_hat = np.empty(len(eval_df))
std_hat = np.empty(len(eval_df))

for i, row in enumerate(eval_df.itertuples(index=False)):
    ctx = {
        "listing_id": row.listing_id,
        "neighborhood_code": row.neighborhood_code,
        "day_of_week": row.day_of_week,
        "is_weekend": row.is_weekend,
        "month": row.month,
    }
    pred = demand_model.predict(ctx, row.price)
    p_hat[i] = pred.prob
    std_hat[i] = pred.std_dev

eval_df["p_hat"] = p_hat
eval_df["std_hat"] = std_hat

print("Demand probabilities & uncertainty precomputed.")


Demand probabilities & uncertainty precomputed.


In [17]:
def fast_prior_predict_factory(base_prob, base_price):
    def predict(X):
        prices = np.expm1(X[:, -1])
        ratio = prices / base_price
        probs = np.clip(base_prob * ratio ** BETA_PRICE, 0.01, 0.99)
        stds = np.full_like(probs, 0.05)
        return probs, stds
    return predict


In [18]:
class OfflineAdapter:
    def __init__(self, bandit):
        self.bandit = bandit

    def set_prior(self, prior_fn):
        self.bandit.predict_fn = prior_fn

    def choose_price(self, context, min_p, max_p):
        return self.bandit.choose_price(context, min_p, max_p)

    def update(self, context, price, booked):
        self.bandit.update(context, price, booked)


Bandits reuse the same feature scaling as the demand model to ensure consistent
geometry between offline priors and online updates.


In [19]:
bandit_feature_names = feature_cols
bandit_scaler = demand_model.scaler


We evaluate multiple pricing policies under identical conditions:

• a historical baseline (no learning)

• Thompson Sampling

• Bayesian UCB

• LinUCB

All policies observe the same contexts and are subject to the same safety constraints.


In [20]:
from pricing_engine import (
    ThompsonPricingBandit,
    BayesianUCBBandit,
    LinUCBBandit,
    PricingDecision,
    SafetyGatedBandit,
    EnterpriseSafeBandit,
)

# -----------------------
# Base bandits
# -----------------------
thompson = OfflineAdapter(
    ThompsonPricingBandit(
        feature_names=feature_cols,
        scaler=demand_model.scaler,
        predict_fn=None,
    )
)

ucb = OfflineAdapter(
    BayesianUCBBandit(
        feature_names=feature_cols,
        scaler=demand_model.scaler,
        predict_fn=None,
        beta=1.5,
    )
)

linucb = OfflineAdapter(
    LinUCBBandit(
        feature_names=feature_cols,
        scaler=demand_model.scaler,
        predict_fn=None,
        alpha=1.0,
    )
)

# -----------------------
# Meta policies (use RAW bandits)
# -----------------------
safety_gated = SafetyGatedBandit(
    thompson.bandit,
    ucb.bandit,
    linucb.bandit,
)

enterprise = EnterpriseSafeBandit(
    ucb.bandit,
    linucb.bandit,
)

# -----------------------
# Historical baseline
# -----------------------
class HistoricalPolicy:
    name = "Historical"

    def choose_price(self, context, **_):
        return PricingDecision(
            selected_price=context["price"],
            expected_revenue=context["price"],
            uncertainty_sigma=0.0,
            source="Historical",
            panic_mode=False,
        )

    def update(self, *args):
        pass


policies = {
    "Historical": HistoricalPolicy(),
    "Thompson": thompson,
    "BayesianUCB": ucb,
    "LinUCB": linucb,
    "SafetyGated": safety_gated,
    "Enterprise": enterprise,
}

print("Policies active:", list(policies.keys()))


Policies active: ['Historical', 'Thompson', 'BayesianUCB', 'LinUCB', 'SafetyGated', 'Enterprise']


Safety constraints are enforced exactly as they would be in production. Policy
outputs are validated and clamped before revenue estimation or learning updates.


In [21]:
from pricing_engine import SafetyGovernor, SafetyConfig

safety = SafetyGovernor(
    SafetyConfig(
        min_price_global=10.0,
        max_price_global=5000.0,
        min_margin_dollars=10.0,
        max_daily_change_pct=0.25,
        uncertainty_penalty_pct=0.15,
    )
)


Historical data is replayed sequentially. For each policy and timestep:

• the policy proposes a price

• demand uncertainty is computed

• the safety governor clamps and smooths the price

• counterfactual revenue is estimated

• the policy updates based on observed outcomes

All safety signals are recorded for downstream analysis.


In [22]:
import time

In [23]:
results = {}
UPDATE_EVERY = 10
for policy_name, policy in policies.items():

    # if policy_name in {"SafetyGated", "Enterprise"}:
    #     continue  # skip meta policies for now

    print(f"\nEvaluating {policy_name}...")

    dr_revenue = 0.0
    prev_price = None

    t0 = time.perf_counter()

    for idx, row in enumerate(eval_df.itertuples(index=False)):

        context = {
            "listing_id": row.listing_id,
            "neighborhood_code": row.neighborhood_code,
            "day_of_week": row.day_of_week,
            "is_weekend": row.is_weekend,
            "month": row.month,
            "price": row.price,
        }

        p_hist = row.price
        y = row.is_booked
        base_prob = row.p_hat
        base_std = row.std_hat

        constraints = {
            "min_price": 0.5 * p_hist,
            "max_price": 3.0 * p_hist,
            "cost_basis": 0.4 * p_hist,
        }

        prior_fn = fast_prior_predict_factory(base_prob, p_hist)

        # Only base bandits receive priors
        if hasattr(policy, "set_prior"):
            policy.set_prior(prior_fn)


        decision = policy.choose_price(
            context,
            min_p=constraints["min_price"],
            max_p=constraints["max_price"],
        )

        safety_res = safety.validate_and_clamp(
            decision.selected_price,
            constraints,
            prev_price,
            {"std_dev": base_std},
        )

        p_safe = safety_res.safe_price
        prev_price = p_safe

        exp_rev = p_safe * base_prob
        if abs(p_safe - p_hist) < 10:
            dr = exp_rev + (p_hist * y - p_hist * base_prob)
        else:
            dr = exp_rev

        dr_revenue += dr

        if idx % UPDATE_EVERY == 0:
            policy.update(context, decision.selected_price, y)


    elapsed = time.perf_counter() - t0

    results[policy_name] = {
        "dr_revenue": dr_revenue,
        "time_sec": elapsed,
        "rows_per_sec": len(eval_df) / elapsed,
    }

    print(
        f"{policy_name} | "
        f"Revenue={dr_revenue:.2f} | "
        f"Time={elapsed:.1f}s | "
        f"Speed={len(eval_df)/elapsed:.0f} rows/sec"
    )



Evaluating Historical...
Historical | Revenue=9582214.12 | Time=2.1s | Speed=130079 rows/sec

Evaluating Thompson...
Thompson | Revenue=14044994.71 | Time=492.2s | Speed=566 rows/sec

Evaluating BayesianUCB...
BayesianUCB | Revenue=16328304.25 | Time=481.8s | Speed=578 rows/sec

Evaluating LinUCB...
LinUCB | Revenue=16442192.70 | Time=46.1s | Speed=6052 rows/sec

Evaluating SafetyGated...
SafetyGated | Revenue=11647719.00 | Time=1066.6s | Speed=261 rows/sec

Evaluating Enterprise...
Enterprise | Revenue=9999908.35 | Time=582.6s | Speed=478 rows/sec


## Safety Dominance, Behavioral Stability & Attribution

Offline revenue uplift alone is insufficient to justify deployment.

A pricing policy may appear profitable while:
• being excessively aggressive,
• relying heavily on safety clamps,
• or violating constraints in edge cases.

In this step, we audit policy behavior under safety governance to ensure that
economic gains are driven by policy intelligence rather than protection layers
or unstable price dynamics.


We evaluate three orthogonal safety and behavior signals:

• Safety clamp rate  
  Fraction of decisions modified by the safety governor.

• Safety violations  
  Any decision that violates hard constraints after governance (must be zero).

• Override rate  
  Fraction of decisions that deviate significantly from historical prices,
  indicating aggressive or unstable behavior.

These metrics separate policy quality from safety intervention.


In [27]:
safety_results = {}

for policy_name, policy in policies.items():

    if policy_name == "Historical":
        continue

    clamps = 0
    overrides = 0
    n = 0

    prev_price = None

    for row in eval_df.itertuples(index=False):

        context = {
            "listing_id": row.listing_id,
            "neighborhood_code": row.neighborhood_code,
            "day_of_week": row.day_of_week,
            "is_weekend": row.is_weekend,
            "month": row.month,
            "price": row.price,
        }

        p_hist = row.price

        constraints = {
            "min_price": 0.5 * p_hist,
            "max_price": 3.0 * p_hist,
            "cost_basis": 0.4 * p_hist,
        }

        decision = policy.choose_price(
            context,
            min_p=constraints["min_price"],
            max_p=constraints["max_price"],
        )

        safety_res = safety.validate_and_clamp(
            decision.selected_price,
            constraints,
            prev_price,
            {"std_dev": row.std_hat},
        )

        p_new = safety_res.safe_price
        prev_price = p_new

        # Safety intervention
        if safety_res.is_clamped:
            clamps += 1

        # Behavioral aggressiveness
        if abs(p_new - p_hist) / p_hist > 0.30:
            overrides += 1

        n += 1

    safety_results[policy_name] = {
        "clamp_rate": clamps / n,
        "override_rate": overrides / n,
    }

    print(
        f"{policy_name} | "
        f"Clamp Rate: {clamps/n:.2%} | "
        f"Override Rate: {overrides/n:.2%}"
    )


Thompson | Clamp Rate: 87.24% | Override Rate: 66.93%
BayesianUCB | Clamp Rate: 80.94% | Override Rate: 68.79%
LinUCB | Clamp Rate: 80.99% | Override Rate: 68.80%
SafetyGated | Clamp Rate: 80.97% | Override Rate: 68.78%
Enterprise | Clamp Rate: 80.98% | Override Rate: 68.78%


Safety metrics are interpreted conservatively:

  • Any safety violation constitutes an automatic NO-GO.

  • High clamp rates indicate that a policy relies on safety to appear reasonable.
  
  • High override rates indicate aggressive deviation from historical pricing and
  require careful scrutiny.

A deployable policy should exhibit low violation counts, moderate clamp rates,
and controlled override behavior.


In [29]:
MAX_CLAMP_RATE = 0.50      # example: safety should not dominate decisions
MAX_OVERRIDE_RATE = 0.40   # example: policy should not be excessively aggressive

SAFETY_OK = all(
    (res["clamp_rate"] <= MAX_CLAMP_RATE) and
    (res["override_rate"] <= MAX_OVERRIDE_RATE)
    for res in safety_results.values()
)

print("Safety gate passed:", SAFETY_OK)


Safety gate passed: False


Beyond safety intervention, we evaluate behavioral stability by examining price
volatility relative to historical pricing.

Excessive volatility may indicate brittle learning dynamics or overreaction to
uncertain demand estimates.


In [30]:
volatility_results = {}

hist_std = eval_df["price"].std()

for policy_name, policy in policies.items():

    if policy_name == "Historical":
        continue

    prices = []

    prev_price = None
    for row in eval_df.itertuples(index=False):
        ctx = {
            "listing_id": row.listing_id,
            "neighborhood_code": row.neighborhood_code,
            "day_of_week": row.day_of_week,
            "is_weekend": row.is_weekend,
            "month": row.month,
            "price": row.price,
        }

        decision = policy.choose_price(
            ctx,
            min_p=0.5 * row.price,
            max_p=3.0 * row.price,
        )

        prices.append(decision.selected_price)

    policy_std = np.std(prices)

    volatility_results[policy_name] = {
        "policy_std": policy_std,
        "volatility_reduction": (hist_std - policy_std) / hist_std
    }

    print(
        f"{policy_name} | "
        f"Price Std: {policy_std:.2f} | "
        f"Volatility Reduction: {volatility_results[policy_name]['volatility_reduction']:.2%}"
    )


Thompson | Price Std: 215.31 | Volatility Reduction: -112.59%
BayesianUCB | Price Std: 218.41 | Volatility Reduction: -115.65%
LinUCB | Price Std: 218.70 | Volatility Reduction: -115.93%
SafetyGated | Price Std: 218.45 | Volatility Reduction: -115.69%
Enterprise | Price Std: 218.44 | Volatility Reduction: -115.68%


Finally, we attribute observed uplift to policy behavior rather than model artifacts
by comparing counterfactual revenue against the historical baseline.

This attribution is conservative: revenue gains must persist after accounting for
safety intervention and stability constraints.


In [ ]:
baseline_revenue = results["Historical"]["dr_revenue"]

for policy_name, res in results.items():
    if policy_name == "Historical":
        continue

    uplift = (res["dr_revenue"] - baseline_revenue) / baseline_revenue

    print(
        f"{policy_name} | "
        f"Uplift vs Baseline: {uplift:.2%} | "
        f"Clamp Rate: {safety_results[policy_name].clamp_rate:.2%} | "
        f"Override Rate: {safety_results[policy_name].override_rate:.2%}"
    )


At this stage, we have established:

• Which policies generate positive counterfactual uplift
• Whether uplift survives safety governance
• Whether policies behave stably and conservatively
• Whether any policy violates hard constraints

Only policies that pass all safety, stability, and attribution checks are eligible
for online shadow testing and stress validation.
